In [56]:
import numpy as np
from collections import defaultdict
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

In [57]:
df = pd.read_csv('/Users/rudxkush/Desktop/Daily Use/Kaggle Dataset/PlayTennis.csv')

In [58]:
df.head()

,Outlook,Temperature,Humidity,Wind,Play Tennis
0,Sunny,Hot,High,Weak,No
1,Sunny,Hot,High,Strong,No
2,Overcast,Hot,High,Weak,Yes
3,Rain,Mild,High,Weak,Yes
4,Rain,Cool,Normal,Weak,Yes


#### Problem 1
$Outlook=Sunny, Temp=Hot, Humidity=High, Wind=Weak → Play Tennis?$ </br>
This is a binary classification problem solved with Naive Bayes:
We compute the posterior for each class and pick the highest.

$P(Play Tennis=Yes | Outlook=Sunny, Temp=Hot, Humidity=High, Wind=Weak)$  is proportional to : </br>
$P(Play Tennis=Yes, Outlook=Sunny) * P(Play Tennis=Yes, Temp=Hot) * P(Play Tennis=Yes, Humidity=High) * P(Play Tennis=Yes, Wind=Weak) * P(Play Tennis=Yes)$

Similarly,
$P(Play Tennis=No | Outlook=Sunny, Temp=Hot, Humidity=High, Wind=Weak)$  is proportional to :  </br>
$P(Play Tennis=No, Outlook=Sunny) * P(Play Tennis=No, Temp=Hot) * P(Play Tennis=No, Humidity=High) * P(Play Tennis=No, Wind=Weak) * P(Play Tennis=No)$

_Predict: whichever posterior is larger wins_

In [59]:
def likelihood(label, feature=None, value=None, alpha=1): # Laplace (add-1) smoothing
    subset = df[df['Play Tennis'] == label]
    if feature is None or value is None:
        return len(subset) / len(df)
    else:
        num_classes = df[feature].nunique()
        return (len(subset[subset[feature] == value]) + alpha) / (len(subset) + alpha * num_classes)

In [60]:
# For Yes
p_sunny_given_yes = likelihood('Yes', feature='Outlook', value='Sunny')
p_hot_given_yes   = likelihood('Yes', feature='Temperature', value='Hot')
p_high_given_yes  = likelihood('Yes', feature='Humidity', value='High')
p_weak_given_yes  = likelihood('Yes', feature='Wind', value='Weak')

# For No
p_sunny_given_no  = likelihood('No', feature='Outlook', value='Sunny')
p_hot_given_no    = likelihood('No', feature='Temperature', value='Hot')
p_high_given_no   = likelihood('No', feature='Humidity', value='High')
p_weak_given_no   = likelihood('No', feature='Wind', value='Weak')

p_yes = likelihood(label='Yes')
p_no = likelihood(label='No')

# Proportional scores
p_yes_given_conditions = p_sunny_given_yes * p_hot_given_yes * p_high_given_yes * p_weak_given_yes * p_yes
p_no_given_conditions  = p_sunny_given_no  * p_hot_given_no  * p_high_given_no  * p_weak_given_no  * p_no

# Print output
if p_yes_given_conditions > p_no_given_conditions:
    print("Suitable to play, Probability = {}%".format(p_yes_given_conditions*100))
else:
    print("Not Suitable to play, Probability = {}%".format(p_no_given_conditions*100))

Not Suitable to play, Probability = 2.0499271137026236%


In [61]:
X = df.iloc[:,:4].values

In [62]:
y = df.iloc[:, -1].values

In [63]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=2
)

In [64]:
class NaiveBayes:
    def fit(self, X, y):
        self.classes = np.unique(y)
        self.priors = {}
        self.likelihoods = {}  # {class: {feature_idx: {value: prob}}}

        for c in self.classes:
            X_c = X[y == c]
            self.priors[c] = len(X_c) / len(y)
            self.likelihoods[c] = {}
            for i in range(X.shape[1]):
                counts = defaultdict(lambda: 1)  # Laplace smoothing
                for val in X_c[:, i]:
                    counts[val] += 1
                total = sum(counts.values())
                self.likelihoods[c][i] = {k: v / total for k, v in counts.items()}

    def predict(self, X):
        return [self._predict_single(x) for x in X]

    def _predict_single(self, x):
        posteriors = {}
        for c in self.classes:
            log_prob = np.log(self.priors[c])
            for i, val in enumerate(x):
                # Use small prob for unseen values (Laplace fallback)
                log_prob += np.log(self.likelihoods[c][i].get(val, 1e-6))
            posteriors[c] = log_prob
        return max(posteriors, key=posteriors.get)

In [65]:
Bayes = NaiveBayes()

In [66]:
Bayes.fit(X_train, y_train)

In [67]:
y_pred = Bayes.predict(X_test)

In [68]:
accuracy_score(y_test, y_pred)

0.6666666666666666

In [69]:
confusion_matrix(y_test, y_pred)

array([[0, 1],
       [0, 2]])

### Note on Model Performance

The accuracy of **66.7%** should be interpreted with caution — the PlayTennis dataset contains only **14 samples**, of which just **3** end up in the test set after an 80/20 split. At this scale, a single misclassification swings accuracy by ~33%.

The model itself is not the bottleneck here. Naive Bayes is well-suited for categorical data like this, but no algorithm can generalise reliably from such a small training set. The confusion matrix reflects this — with only 3 test samples, it carries almost no statistical meaning.

For a meaningful evaluation, a dataset of at least a few hundred samples would be needed, or alternatively, **k-fold cross-validation** could be used here to make better use of the 14 available samples rather than a fixed train/test split.